# Competição de Modelos — California Housing Dataset

> 📄 Documentação detalhada: [`model_competition_notes.md`](model_competition_notes.md)

---

| Parâmetro | Valor |
|-----------|-------|
| Dataset | California Housing (sklearn) |
| Target | `ValorMedioResidencias` (log1p) |
| Modelos | Ridge · XGBoost · LightGBM · CatBoost · TabNet |
| Otimização | Optuna (TPE Bayesiano) com persistência SQLite |

---
## 1. Configuração do Ambiente

### 1.1 Imports e Dependências

In [ ]:
import sys, json, warnings
from pathlib import Path
from datetime import datetime
import webbrowser

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy.stats import skew

import sklearn
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin, RegressorMixin
from sklearn.linear_model import Ridge
from sklearn.metrics import (
    r2_score, mean_absolute_error,
    mean_squared_error, mean_absolute_percentage_error
)
import joblib

try:
    import xgboost as xgb
    print(f"XGBoost  : {xgb.__version__}")
except ImportError:
    raise ImportError("Execute: pip install xgboost")

try:
    import lightgbm as lgb
    print(f"LightGBM : {lgb.__version__}")
except ImportError:
    raise ImportError("Execute: pip install lightgbm")

try:
    import catboost as cb
    print(f"CatBoost : {cb.__version__}")
except ImportError:
    raise ImportError("Execute: pip install catboost")

try:
    import torch
    from pytorch_tabnet.tab_model import TabNetRegressor
    print(f"PyTorch  : {torch.__version__}")
    print(f"TabNet   : pytorch-tabnet instalado")
except ImportError:
    raise ImportError("Execute: pip install pytorch-tabnet torch")

try:
    import optuna
    from optuna.importance import get_param_importances
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    print(f"Optuna   : {optuna.__version__}")
except ImportError:
    raise ImportError("Execute: pip install optuna")

# ── Detecção de GPU ────────────────────────────────────────────────────────
CUDA_AVAILABLE = torch.cuda.is_available()
DEVICE         = 'cuda' if CUDA_AVAILABLE else 'cpu'

if CUDA_AVAILABLE:
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"\nGPU detectada : {gpu_name} ({gpu_mem:.1f} GB VRAM)")
    print("Uso de GPU    : TabNet (treino neural), CatBoost (task_type=GPU), XGBoost (device=cuda)")
    print("NOTA          : Para ~12k amostras, GPU em árvores pode ser comparável ao CPU.")
    print("                GPU é mais vantajosa a partir de ~100k+ linhas para tree boosting.")
else:
    print("\nGPU/CUDA não detectada — todos os modelos usarão CPU.")

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid')
plt.rcParams.update({'figure.dpi': 100})

RANDOM_STATE = 42
ARTIFACT_DIR = Path('artifacts/')
ARTIFACT_DIR.mkdir(exist_ok=True)

print(f"\nscikit-learn : {sklearn.__version__}")
print(f"Python       : {sys.version.split()[0]}")

### 1.2 Métricas de Avaliação

Métricas de ponto (R², RMSE, MAE) para o Round 1 e métricas de confiabilidade (IS, PICP, MPIW, MACE) para o Round 2. Ver `model_competition_notes.md §1.2`.

In [ ]:
# ─────────────────────────────────────────────────────────────
# Funções de métricas de ponto e confiabilidade
# ─────────────────────────────────────────────────────────────

def metricas_ponto(y_true_log, y_pred_log, label=''):
    """Métricas de ponto — R², RMSE, MAE, MAPE na escala original ($100k)."""
    y_true = np.expm1(y_true_log)
    y_pred = np.expm1(y_pred_log)
    return {
        'Modelo'      : label,
        'R²'          : r2_score(y_true_log, y_pred_log),
        'RMSE ($100k)': np.sqrt(mean_squared_error(y_true, y_pred)),
        'MAE ($100k)' : mean_absolute_error(y_true, y_pred),
        'MAPE (%)'    : mean_absolute_percentage_error(y_true, y_pred) * 100,
    }


def interval_score(y_true, lower, upper, alpha=0.2):
    """
    Interval Score (IS) — regra de pontuação própria para intervalos de predição.

    Análogo ao Brier Score para regressão. Penaliza:
    - Intervalos largos (ineficiência)
    - Valores fora do intervalo (má cobertura)

    IS_α = (u−l) + (2/α)·max(l−y, 0) + (2/α)·max(y−u, 0)

    Menor = melhor. alpha = 1 − confidence_level.
    """
    width         = upper - lower
    penalty_low   = (2 / alpha) * np.maximum(lower - y_true, 0)
    penalty_high  = (2 / alpha) * np.maximum(y_true - upper, 0)
    return float(np.mean(width + penalty_low + penalty_high))


def picp(y_true, lower, upper):
    """Prediction Interval Coverage Probability."""
    return float(np.mean((y_true >= lower) & (y_true <= upper)))


def mpiw(lower, upper):
    """Mean Prediction Interval Width."""
    return float(np.mean(upper - lower))


def conformal_intervals(pipeline, X_cal, y_cal_log, X_test, confidence=0.80):
    """
    Split Conformal Prediction — intervalos com garantia de cobertura.

    Parâmetros
    ----------
    pipeline    : modelo sklearn já treinado em X_train
    X_cal       : conjunto de calibração (nunca visto pelo pipeline.fit)
    y_cal_log   : target real do conjunto de calibração (log-scale)
    X_test      : conjunto de teste
    confidence  : nível de confiança nominal (ex: 0.80 = 80%)

    Retorna (em log-scale)
    -------
    y_pred_log, lower_log, upper_log, q_hat
    """
    alpha = 1 - confidence
    n_cal = len(y_cal_log)

    y_cal_pred  = pipeline.predict(X_cal)
    scores      = np.abs(np.asarray(y_cal_log) - y_cal_pred)

    # Quantil com correção de cobertura finita
    q_level = min(1.0, (1 - alpha) * (1 + 1 / n_cal))
    q_hat   = float(np.quantile(scores, q_level))

    y_pred  = pipeline.predict(X_test)
    return y_pred, y_pred - q_hat, y_pred + q_hat, q_hat


def mace(pipeline, X_cal, y_cal_log, X_test, y_test_log, levels=None):
    """
    Mean Absolute Calibration Error — análogo ao ECE da classificação.

    Computa a cobertura empírica em múltiplos níveis de confiança e
    mede o desvio médio em relação à cobertura nominal.

    Menor MACE = intervalos mais bem calibrados.
    """
    if levels is None:
        levels = np.arange(0.10, 1.00, 0.10)

    y_true = np.asarray(y_test_log)
    errors = []
    for conf in levels:
        _, lower, upper, _ = conformal_intervals(pipeline, X_cal, y_cal_log, X_test, confidence=conf)
        cov_empirica = picp(y_true, lower, upper)
        errors.append(abs(cov_empirica - conf))

    return float(np.mean(errors))


print("Funções de métricas definidas: metricas_ponto, interval_score, picp, mpiw, conformal_intervals, mace")

---
## 2. Preparação do Pipeline

### 2.1 Transformadores Customizados

In [ ]:
# ─────────────────────────────────────────────────────────────
# Transformadores customizados (notebook auto-suficiente)
# ─────────────────────────────────────────────────────────────
from sklearn.utils.validation import check_is_fitted


class WinsorizacaoTransformer(BaseEstimator, TransformerMixin):
    """Winsorização com bounds IQR aprendidos no treino (sem data leakage)."""

    def __init__(self, colunas=None, k=3.0):
        self.colunas = colunas
        self.k = k

    def fit(self, X, y=None):
        df = pd.DataFrame(X) if not isinstance(X, pd.DataFrame) else X
        cols = self.colunas or df.columns.tolist()
        self.bounds_ = {}
        for col in cols:
            if col in df.columns:
                q1, q3 = df[col].quantile([0.25, 0.75])
                iqr = q3 - q1
                self.bounds_[col] = (q1 - self.k * iqr, q3 + self.k * iqr)
        return self

    def transform(self, X):
        check_is_fitted(self, attributes=['bounds_'])
        df = pd.DataFrame(X).copy() if not isinstance(X, pd.DataFrame) else X.copy()
        for col, (lo, hi) in self.bounds_.items():
            if col in df.columns:
                df[col] = df[col].clip(lo, hi)
        return df


class CaliforniaHousingTransformer(BaseEstimator, TransformerMixin):
    """
    Feature engineering para o California Housing Dataset.
    Log1p em features assimétricas + ratios + distâncias geográficas (Euclidiana em graus).
    """

    _CITIES = {
        'sf': (37.7749, -122.4194),
        'la': (34.0522, -118.2437),
        'sd': (32.7157, -117.1611),
    }
    LOG1P_FEATURES = ['RendaMediana', 'Populacao', 'MediaOcupacao']
    INPUT_COLS = [
        'RendaMediana', 'IdadeMediaResidencias', 'MediaComodos',
        'MediaQuartos', 'Populacao', 'MediaOcupacao', 'Latitude', 'Longitude',
    ]
    OUTPUT_COLS = INPUT_COLS + [
        'razao_quartos', 'comodos_por_pessoa',
        'dist_sf', 'dist_la', 'dist_sd',
    ]

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        df = X[self.INPUT_COLS].copy() if isinstance(X, pd.DataFrame) \
             else pd.DataFrame(X, columns=self.INPUT_COLS[:X.shape[1]])

        for col in self.LOG1P_FEATURES:
            df[col] = np.log1p(df[col])

        df['razao_quartos']      = df['MediaQuartos']  / (df['MediaComodos']  + 1e-8)
        df['comodos_por_pessoa'] = df['MediaComodos']  / (df['MediaOcupacao'] + 1e-8)

        for city, (lat, lon) in self._CITIES.items():
            df[f'dist_{city}'] = np.sqrt(
                (df['Latitude']  - lat) ** 2 +
                (df['Longitude'] - lon) ** 2
            )
        return df[self.OUTPUT_COLS].values

    def get_feature_names_out(self, input_features=None):
        return np.array(self.OUTPUT_COLS)


COLS_WINSOR = ['MediaComodos', 'MediaQuartos', 'Populacao', 'MediaOcupacao']
print(f"Transformadores definidos.")
print(f"Features de saída ({len(CaliforniaHousingTransformer.OUTPUT_COLS)}): {CaliforniaHousingTransformer.OUTPUT_COLS}")

### 2.2 Wrapper TabNet (Compatibilidade sklearn)

In [ ]:
# ─────────────────────────────────────────────────────────────
# Wrapper sklearn-compatível para TabNet
# ─────────────────────────────────────────────────────────────

class TabNetRegressorWrapper(BaseEstimator, RegressorMixin):
    """
    Encapsula pytorch-tabnet.TabNetRegressor em interface sklearn.

    Necessário porque TabNetRegressor não aceita fit(X, y) puro na Pipeline:
    exige arrays numpy e parâmetros de treinamento (epochs, patience, etc.)
    que precisam ser passados no __init__ para ser compatível com Pipeline.

    GPU: usa DEVICE global se CUDA disponível.

    NOTAS sobre TabNet neste dataset:
    - California Housing (~12k linhas, 11 features) é pequeno para redes neurais.
    - TabNet costuma ficar abaixo de LightGBM/XGBoost/CatBoost em dados tabulares
      pequenos, mas oferece explicabilidade via máscaras de atenção.
    - Se for o vencedor, interprete com cautela — possivelmente overfitou.
    """

    def __init__(
        self,
        n_d=32, n_a=32, n_steps=3, gamma=1.3,
        n_independent=2, n_shared=2,
        lambda_sparse=1e-3, momentum=0.02,
        max_epochs=300, patience=30,
        batch_size=256, virtual_batch_size=128,
        val_ratio=0.15, verbose=0,
    ):
        self.n_d = n_d; self.n_a = n_a; self.n_steps = n_steps
        self.gamma = gamma; self.n_independent = n_independent
        self.n_shared = n_shared; self.lambda_sparse = lambda_sparse
        self.momentum = momentum; self.max_epochs = max_epochs
        self.patience = patience; self.batch_size = batch_size
        self.virtual_batch_size = virtual_batch_size
        self.val_ratio = val_ratio; self.verbose = verbose

    def fit(self, X, y):
        X_arr = np.asarray(X, dtype=np.float32)
        y_arr = np.asarray(y, dtype=np.float32).reshape(-1, 1)
        n_val = max(1, int(len(X_arr) * self.val_ratio))
        X_tr, X_val = X_arr[:-n_val], X_arr[-n_val:]
        y_tr, y_val = y_arr[:-n_val], y_arr[-n_val:]
        self.model_ = TabNetRegressor(
            n_d=self.n_d, n_a=self.n_a, n_steps=self.n_steps, gamma=self.gamma,
            n_independent=self.n_independent, n_shared=self.n_shared,
            lambda_sparse=self.lambda_sparse, momentum=self.momentum,
            device_name=DEVICE, seed=RANDOM_STATE, verbose=self.verbose,
        )
        self.model_.fit(
            X_tr, y_tr, eval_set=[(X_val, y_val)],
            eval_name=['val'], eval_metric=['mae'],
            max_epochs=self.max_epochs, patience=self.patience,
            batch_size=self.batch_size, virtual_batch_size=self.virtual_batch_size,
        )
        return self

    def predict(self, X):
        return self.model_.predict(np.asarray(X, dtype=np.float32)).ravel()


print("TabNetRegressorWrapper definido.")

---
## 3. Dados

### 3.1 Carregamento e Divisão

Divisão 3-way: 60 % treino / 20 % calibração (conformal) / 20 % teste.

In [ ]:
# ─────────────────────────────────────────────────────────────
# Carregamento dos dados + divisão 3-way
# ─────────────────────────────────────────────────────────────
housing = fetch_california_housing(as_frame=True)
df_raw = housing.frame.rename(columns={
    'MedInc'     : 'RendaMediana',
    'HouseAge'   : 'IdadeMediaResidencias',
    'AveRooms'   : 'MediaComodos',
    'AveBedrms'  : 'MediaQuartos',
    'Population' : 'Populacao',
    'AveOccup'   : 'MediaOcupacao',
    'MedHouseVal': 'ValorMedioResidencias',
})

FEATURES = [c for c in df_raw.columns if c != 'ValorMedioResidencias']
TARGET   = 'ValorMedioResidencias'

X = df_raw[FEATURES].copy()
y = np.log1p(df_raw[TARGET])   # log1p na target

# Divisão estratificada em 3 partes (conformal prediction exige conjunto de calibração separado)
#   Treino (fit):   60% do total — para treinar o modelo
#   Calibração:     20% do total — para calibrar os intervalos (conformal)
#   Teste:          20% do total — avaliação final (intocável)

X_trainval, X_test,   y_trainval, y_test   = train_test_split(X, y,          test_size=0.20, random_state=RANDOM_STATE)
X_train,    X_cal,    y_train,    y_cal    = train_test_split(X_trainval, y_trainval, test_size=0.25, random_state=RANDOM_STATE)

print("Divisão 3-way:")
print(f"  X_train    : {X_train.shape[0]:>6,} amostras  (60% do total)")
print(f"  X_cal      : {X_cal.shape[0]:>6,} amostras  (20% — calibração conformal)")
print(f"  X_test     : {X_test.shape[0]:>6,} amostras  (20% — avaliação final)")
print(f"\nTarget: log1p(ValorMedioResidencias)")
print(f"  y_test: min={y_test.min():.3f}, max={y_test.max():.3f}, mean={y_test.mean():.3f}")

### 3.2 Factory de Pipelines

```
Winsorização  →  Feature Engineering  →  StandardScaler  →  Modelo
```

In [ ]:
def criar_pipeline(modelo) -> Pipeline:
    """Pipeline completo: Winsorização → Feature Eng → Scaler → Modelo."""
    return Pipeline([
        ('winsorizacao', WinsorizacaoTransformer(colunas=COLS_WINSOR, k=3.0)),
        ('transformer',  CaliforniaHousingTransformer()),
        ('scaler',       StandardScaler()),
        ('modelo',       modelo),
    ])

print("Factory criar_pipeline() definida.")

---
## 4. Competidores

### 4.1 Descrição dos Modelos

Ridge · XGBoost · LightGBM · CatBoost · TabNet. Ver `model_competition_notes.md §4`.

---
## 5. Round 1 — Acurácia de Ponto

### 5.1 Baseline: CV com Parâmetros Padrão

In [ ]:
# Comparação inicial com hiperparâmetros padrão e poucas epocas (antes do tuning)
kfold = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

_xgb_device   = {'device': 'cuda'} if CUDA_AVAILABLE else {}
_cat_task_type = 'GPU' if CUDA_AVAILABLE else 'CPU'

MODELOS_DEFAULT = {
    'Ridge'   : Ridge(alpha=1.0),
    'XGBoost' : xgb.XGBRegressor(random_state=RANDOM_STATE, verbosity=0, n_jobs=-1, **_xgb_device),
    'LightGBM': lgb.LGBMRegressor(random_state=RANDOM_STATE, verbose=-1, n_jobs=-1),
    'CatBoost': cb.CatBoostRegressor(random_state=RANDOM_STATE, verbose=0, task_type=_cat_task_type),
    'TabNet'  : TabNetRegressorWrapper(max_epochs=30, patience=5, verbose=0),
}

print("Comparação inicial — hiperparâmetros default (X_train, 5-fold CV):")
print(f"{'Modelo':<12} {'R² Médio':>10} {'R² Std':>8}")
print("-" * 33)
resultados_default = {}
for nome, modelo in MODELOS_DEFAULT.items():
    pipe   = criar_pipeline(modelo)
    scores = cross_val_score(pipe, X_train, y_train, cv=kfold, scoring='r2', n_jobs=1)
    resultados_default[nome] = scores
    print(f"{nome:<12} {scores.mean():>10.4f} {scores.std():>8.4f}")

### 5.2 Otimização de Hiperparâmetros com Optuna

#### 5.2.1 Estratégia de Busca

TPE Bayesiano com `multivariate=True` + MedianPruner para boosting. Ver `model_competition_notes.md §5.2`.

In [ ]:
import socket, subprocess, time, pathlib, sys, urllib.parse

# ─── Banco SQLite — salvo na pasta do notebook ────────────────────────────
_NOTEBOOK_DIR  = pathlib.Path.cwd()
OPTUNA_DB_PATH = _NOTEBOOK_DIR / "california_housing_optuna.db"
STORAGE        = f"sqlite:///{OPTUNA_DB_PATH}"
DASHBOARD_PORT = 8080

_DASHBOARD_EXE = pathlib.Path(sys.executable).parent / "optuna-dashboard"

print(f"Banco de estudos : {OPTUNA_DB_PATH}")
print(f"Dashboard URL    : http://localhost:{DASHBOARD_PORT}")


def _port_in_use(port: int) -> bool:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        return s.connect_ex(("localhost", port)) == 0


def launch_optuna_dashboard(
    db_path: pathlib.Path = OPTUNA_DB_PATH,
    port: int = DASHBOARD_PORT,
    height: int = 750,
) -> None:
    """Inicia optuna-dashboard e exibe embutido na saida da celula (VS Code / Jupyter)."""
    url = f"http://localhost:{port}"

    # Inicia servidor se nao estiver rodando
    if not _port_in_use(port):
        subprocess.Popen(
            [str(_DASHBOARD_EXE), f"sqlite:///{db_path}",
             "--port", str(port), "--host", "localhost"],
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL,
            creationflags=subprocess.CREATE_NEW_PROCESS_GROUP,
        )
        time.sleep(2.0)
        print(f"Servidor iniciado — {url}")
    else:
        print(f"Servidor ja rodando — {url}")

    # # Exibe embutido na saida da celula (funciona no VS Code e no Jupyter classic)
    # from IPython.display import display, IFrame
    # display(IFrame(src=url, width="100%", height=height))

    # Alternativa: abrir no navegador padrao do sistema
    webbrowser.open(url)


#### Abrir Dashboard Optuna

Rode a célula abaixo a qualquer momento para visualizar os estudos — mesmo sem re-executar o tuning.

> **Nota:** a célula seguinte é do tipo `raw` intencionalmente — evita abertura automática do browser ao executar `nbconvert --execute`. Execute-a manualmente no Jupyter quando quiser inspecionar os trials.

#### 5.2.2 Execução da Busca

Estudos persistidos em SQLite — re-rodar acumula trials.

In [ ]:
# ─── Configuração global do Optuna ───────────────────────────────────────
PRUNER_BOOSTING = optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=1, interval_steps=1)

# ─── Metas totais de trials por modelo (edite conforme necessário) ────────
TARGET_TRIALS = {
    "CalHousing_Ridge":    500,
    "CalHousing_XGBoost":  500,
    "CalHousing_LightGBM": 500,
    "CalHousing_CatBoost": 500,
    "CalHousing_TabNet":   100,
}

def _remaining(study_name: str) -> int:
    """Retorna quantos trials ainda faltam para atingir o TARGET."""
    target = TARGET_TRIALS[study_name]
    try:
        s = optuna.load_study(study_name=study_name, storage=STORAGE)
        done = sum(1 for t in s.trials if t.state == optuna.trial.TrialState.COMPLETE)
    except Exception:
        done = 0
    remaining = max(0, target - done)
    print(f"  {study_name}: {done} completos, meta {target} → rodando {remaining}")
    return remaining

N_TRIALS_RIDGE    = _remaining("CalHousing_Ridge")
N_TRIALS_XGBOOST  = _remaining("CalHousing_XGBoost")
N_TRIALS_LIGHTGBM = _remaining("CalHousing_LightGBM")
N_TRIALS_CATBOOST = _remaining("CalHousing_CatBoost")
N_TRIALS_TABNET   = _remaining("CalHousing_TabNet")

kfold_optuna = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)


def _mk_sampler():
    """Cria TPESampler fresco — cada estudo recebe sua própria instância."""
    return optuna.samplers.TPESampler(
        consider_prior          = True,
        prior_weight            = 1.0,
        consider_magic_clip     = True,
        consider_endpoints      = False,
        n_startup_trials        = 50,
        n_ei_candidates         = 20,
        seed                    = RANDOM_STATE,
        multivariate            = True,
        group                   = True,
        warn_independent_sampling = True,
        constant_liar           = False,
    )


def objetivo_ridge(trial):
    alpha = trial.suggest_float("alpha", 1e-4, 1e4, log=True)
    pipe  = criar_pipeline(Ridge(alpha=alpha))
    return float(cross_val_score(pipe, X_train, y_train, cv=kfold_optuna, scoring="r2", n_jobs=-1).mean())


def _cv_com_poda(trial, modelo_cls, hparams):
    scores = []
    for step, (tr_idx, val_idx) in enumerate(kfold_optuna.split(X_train)):
        pipe = criar_pipeline(modelo_cls(**hparams))
        pipe.fit(X_train.iloc[tr_idx], y_train.iloc[tr_idx])
        scores.append(float(r2_score(y_train.iloc[val_idx], pipe.predict(X_train.iloc[val_idx]))))
        trial.report(float(np.mean(scores)), step)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()
    return float(np.mean(scores))


def objetivo_xgboost(trial):
    hparams = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 600),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.30, log=True),
        "max_depth": trial.suggest_int("max_depth", 3, 9),
        "subsample": trial.suggest_float("subsample", 0.50, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.50, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        "gamma": trial.suggest_float("gamma", 1e-8, 1.0, log=True),
        "random_state": RANDOM_STATE, "verbosity": 0, "n_jobs": 1,
        **({"device": "cuda"} if CUDA_AVAILABLE else {}),
    }
    return _cv_com_poda(trial, xgb.XGBRegressor, hparams)


def objetivo_lightgbm(trial):
    hparams = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 600),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.30, log=True),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "num_leaves": trial.suggest_int("num_leaves", 20, 300),
        "subsample": trial.suggest_float("subsample", 0.50, 1.0),
        "subsample_freq": trial.suggest_int("subsample_freq", 0, 7),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.50, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 100),
        "random_state": RANDOM_STATE, "verbose": -1, "n_jobs": 1,
    }
    return _cv_com_poda(trial, lgb.LGBMRegressor, hparams)


def objetivo_catboost(trial):
    hparams = {
        "iterations": trial.suggest_int("iterations", 200, 800),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.30, log=True),
        "depth": trial.suggest_int("depth", 4, 10),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1e-3, 10.0, log=True),
        "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 1.0),
        "random_strength": trial.suggest_float("random_strength", 1e-3, 10.0, log=True),
        "border_count": trial.suggest_int("border_count", 32, 255),
        "random_state": RANDOM_STATE, "verbose": 0, "task_type": _cat_task_type,
    }
    return _cv_com_poda(trial, cb.CatBoostRegressor, hparams)


def objetivo_tabnet(trial):
    hparams = {
        "n_d": trial.suggest_int("n_d", 8, 64),
        "n_a": trial.suggest_int("n_a", 8, 64),
        "n_steps": trial.suggest_int("n_steps", 2, 6),
        "gamma": trial.suggest_float("gamma", 1.0, 2.0),
        "lambda_sparse": trial.suggest_float("lambda_sparse", 1e-6, 1e-2, log=True),
        "momentum": trial.suggest_float("momentum", 0.01, 0.4),
        "max_epochs": 300, "patience": 30,
        "batch_size": trial.suggest_categorical("batch_size", [128, 256, 512]),
        "virtual_batch_size": 128, "verbose": 0,
    }
    kf3 = KFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
    scores = []
    for tr_idx, val_idx in kf3.split(X_train):
        pipe = criar_pipeline(TabNetRegressorWrapper(**hparams))
        pipe.fit(X_train.iloc[tr_idx], y_train.iloc[tr_idx])
        scores.append(float(r2_score(y_train.iloc[val_idx], pipe.predict(X_train.iloc[val_idx]))))
    return float(np.mean(scores))


# ─── Helper: cria/retoma estudo e executa otimização ─────────────────
def _run_study(study_name, sampler, pruner, objetivo_fn, n_trials):
    study = optuna.create_study(
        study_name=study_name,
        storage=STORAGE,
        direction="maximize",
        sampler=sampler,
        pruner=pruner,
        load_if_exists=True,
    )
    n_done = sum(1 for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE)
    if n_done > 0:
        print(f"  Retomando: {n_done} trials existentes -> adicionando mais {n_trials}")
    study.optimize(objetivo_fn, n_trials=n_trials, show_progress_bar=True)
    return study


# ─── Abrir dashboard antes de iniciar ────────────────────────────────────
launch_optuna_dashboard()

# ─── Execução dos estudos ─────────────────────────────────────────────
print(f"\n[1/5] Otimizando Ridge ({N_TRIALS_RIDGE} trials)...")
study_ridge = _run_study("CalHousing_Ridge", _mk_sampler(), optuna.pruners.NopPruner(), objetivo_ridge, N_TRIALS_RIDGE)
print(f"      Melhor R2={study_ridge.best_value:.4f}")

print(f"\n[2/5] Otimizando XGBoost ({N_TRIALS_XGBOOST} trials)...")
study_xgb = _run_study("CalHousing_XGBoost", _mk_sampler(), PRUNER_BOOSTING, objetivo_xgboost, N_TRIALS_XGBOOST)
n_pruned_xgb = sum(1 for t in study_xgb.trials if t.state == optuna.trial.TrialState.PRUNED)
print(f"      Melhor R2={study_xgb.best_value:.4f} | podados: {n_pruned_xgb}")

print(f"\n[3/5] Otimizando LightGBM ({N_TRIALS_LIGHTGBM} trials)...")
study_lgb = _run_study("CalHousing_LightGBM", _mk_sampler(), PRUNER_BOOSTING, objetivo_lightgbm, N_TRIALS_LIGHTGBM)
n_pruned_lgb = sum(1 for t in study_lgb.trials if t.state == optuna.trial.TrialState.PRUNED)
print(f"      Melhor R2={study_lgb.best_value:.4f} | podados: {n_pruned_lgb}")

print(f"\n[4/5] Otimizando CatBoost ({N_TRIALS_CATBOOST} trials)...")
study_cat = _run_study("CalHousing_CatBoost", _mk_sampler(), PRUNER_BOOSTING, objetivo_catboost, N_TRIALS_CATBOOST)
n_pruned_cat = sum(1 for t in study_cat.trials if t.state == optuna.trial.TrialState.PRUNED)
print(f"      Melhor R2={study_cat.best_value:.4f} | podados: {n_pruned_cat}")

print(f"\n[5/5] Otimizando TabNet ({N_TRIALS_TABNET} trials, 3-fold)...")
study_tab = _run_study("CalHousing_TabNet", _mk_sampler(), optuna.pruners.NopPruner(), objetivo_tabnet, N_TRIALS_TABNET)
print(f"      Melhor R2={study_tab.best_value:.4f}")

# ─── Construir modelos finais ─────────────────────────────────────────
melhor_ridge = Ridge(alpha=study_ridge.best_params["alpha"])
melhor_xgb = xgb.XGBRegressor(**study_xgb.best_params, random_state=RANDOM_STATE, verbosity=0, n_jobs=-1,
                               **({"device": "cuda"} if CUDA_AVAILABLE else {}))
melhor_lgb = lgb.LGBMRegressor(**study_lgb.best_params, random_state=RANDOM_STATE, verbose=-1, n_jobs=-1)
melhor_cat = cb.CatBoostRegressor(**study_cat.best_params, random_state=RANDOM_STATE, verbose=0,
                                   task_type=_cat_task_type)
melhor_tab = TabNetRegressorWrapper(**study_tab.best_params)

study_map = {"Ridge": study_ridge, "XGBoost": study_xgb, "LightGBM": study_lgb,
             "CatBoost": study_cat, "TabNet": study_tab}
model_map  = {"Ridge": melhor_ridge, "XGBoost": melhor_xgb, "LightGBM": melhor_lgb,
              "CatBoost": melhor_cat, "TabNet": melhor_tab}

pipelines_tunados = {nome: criar_pipeline(modelo) for nome, modelo in model_map.items()}
print("\nOptuna concluido. Pipelines prontos.")

#### 5.2.3 Visualização dos Estudos Optuna

In [ ]:
# ─── Visualização dos estudos Optuna — 5 modelos ─────────────────────────
CORES = {
    'Ridge': '#e74c3c', 'XGBoost': '#3498db', 'LightGBM': '#2ecc71',
    'CatBoost': '#9b59b6', 'TabNet': '#f39c12',
}
estudos_viz = [(n, study_map[n], CORES[n]) for n in CORES]

fig, axes = plt.subplots(2, 5, figsize=(26, 10))

for col, (nome, study, cor) in enumerate(estudos_viz):
    df_trials  = study.trials_dataframe()
    concluidos = df_trials[df_trials['state'] == 'COMPLETE'].copy()

    ax_hist = axes[0, col]
    ax_hist.scatter(concluidos['number'], concluidos['value'], alpha=0.4, s=12, c=cor)
    ax_hist.plot(concluidos['number'], concluidos['value'].cummax(), c='black', lw=2)
    ax_hist.axhline(study.best_value, color='red', linestyle='--', lw=1, alpha=0.7)
    ax_hist.set_title(f'{nome}\nMelhor R²={study.best_value:.4f}', fontsize=10)
    ax_hist.set_xlabel('Trial #'); ax_hist.set_ylabel('R² (CV)')
    ax_hist.grid(True, alpha=0.3)

    ax_imp = axes[1, col]
    try:
        importances = get_param_importances(study)
        params_sorted = sorted(importances.items(), key=lambda x: x[1])
        param_names = [p for p, _ in params_sorted]
        param_vals  = [v for _, v in params_sorted]
        bar_colors  = [cor if v == max(param_vals) else '#bdc3c7' for v in param_vals]
        ax_imp.barh(param_names, param_vals, color=bar_colors, alpha=0.85, edgecolor='white')
        ax_imp.set_xlabel('Importância (Fanova)', fontsize=8)
        ax_imp.tick_params(axis='y', labelsize=7)
    except Exception:
        ax_imp.text(0.5, 0.5, 'Importância\nnão disponível', ha='center', va='center',
                    transform=ax_imp.transAxes, fontsize=10)
    ax_imp.set_title(f'{nome} — Hiperparâmetros', fontsize=9)
    ax_imp.grid(True, axis='x', alpha=0.3)

plt.suptitle('Estudos Optuna — Histórico e Importância de Hiperparâmetros (5 modelos)',
             fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

print("\nMelhores hiperparâmetros:")
for nome, study, _ in estudos_viz:
    print(f"\n  {nome}:")
    for k, v in study.best_params.items():
        print(f"    {k}: {v}")

### 5.3 Resultados do Round 1

#### 5.3.1 Métricas de CV (5-Fold)

In [ ]:
# ─── Round 1: CV 5-Fold nos 5 modelos ────────────────────────────────────
resultados_r1 = []
for nome, pipe in pipelines_tunados.items():
    r2_cv   = cross_val_score(pipe, X_train, y_train, cv=kfold, scoring='r2', n_jobs=1)
    rmse_cv = cross_val_score(pipe, X_train, y_train, cv=kfold, scoring='neg_root_mean_squared_error', n_jobs=1)
    mae_cv  = cross_val_score(pipe, X_train, y_train, cv=kfold, scoring='neg_mean_absolute_error', n_jobs=1)
    mape_cv = cross_val_score(pipe, X_train, y_train, cv=kfold, scoring='neg_mean_absolute_percentage_error', n_jobs=1)
    resultados_r1.append({
        'Modelo'     : nome,
        'R² (CV)'    : r2_cv.mean(),
        'R² Std'     : r2_cv.std(),
        'RMSE (CV)'  : -rmse_cv.mean(),
        'MAE (CV)'   : -mae_cv.mean(),
        'MAPE % (CV)': -mape_cv.mean() * 100,
    })

df_r1 = pd.DataFrame(resultados_r1).set_index('Modelo')

print("=" * 65)
print("ROUND 1 — MÉTRICAS DE PONTO (Cross-Validation 5-Fold no treino)")
print("=" * 65)
display(df_r1.round(4))

bar_colors = [CORES[n] for n in df_r1.index]
fig, axes = plt.subplots(1, 4, figsize=(22, 5))
for ax, col in zip(axes, ['R² (CV)', 'RMSE (CV)', 'MAE (CV)', 'MAPE % (CV)']):
    vals = df_r1[col]
    bars = ax.bar(df_r1.index, vals, color=bar_colors, alpha=0.88, edgecolor='white', width=0.6)
    ax.set_title(col, fontsize=11)
    ax.set_xticklabels(df_r1.index, rotation=30, ha='right', fontsize=9)
    if col == 'R² (CV)':
        ax.set_ylim(0, 1)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(vals)*0.015,
                f'{v:.3f}', ha='center', fontsize=8)
patches = [plt.Rectangle((0,0),1,1, color=CORES[n], alpha=0.88) for n in df_r1.index]
axes[0].legend(patches, df_r1.index, fontsize=8, title='Modelo')
plt.suptitle("Round 1 — Métricas de Ponto (CV 5-Fold)", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

#### 5.3.2 Diagnóstico de Overfitting — Treino vs. Teste

Gap R² < −2 % = verde | −2 % a −5 % = laranja | > −5 % = vermelho.

In [ ]:
# ─── Comparação Treino vs Teste — Diagnóstico de Overfitting ─────────────
print("Treinando e avaliando em treino + teste...")
metricas_treino, metricas_teste = [], []

for nome, pipe in pipelines_tunados.items():
    pipe.fit(X_train, y_train)
    pred_tr = pipe.predict(X_train)
    pred_te = pipe.predict(X_test)
    metricas_treino.append({'Modelo': nome,
        'R²': r2_score(y_train, pred_tr), 'RMSE': np.sqrt(mean_squared_error(y_train, pred_tr)),
        'MAE': mean_absolute_error(y_train, pred_tr)})
    metricas_teste.append({'Modelo': nome,
        'R²': r2_score(y_test, pred_te), 'RMSE': np.sqrt(mean_squared_error(y_test, pred_te)),
        'MAE': mean_absolute_error(y_test, pred_te)})

df_treino = pd.DataFrame(metricas_treino).set_index('Modelo')
df_teste  = pd.DataFrame(metricas_teste).set_index('Modelo')

df_comp = pd.concat([df_treino.add_suffix(' (Treino)'), df_teste.add_suffix(' (Teste)')], axis=1)[
    ['R² (Treino)', 'R² (Teste)', 'RMSE (Treino)', 'RMSE (Teste)', 'MAE (Treino)', 'MAE (Teste)']]
df_comp['Gap R² (Teste−Treino)'] = df_teste['R²'] - df_treino['R²']

print("\n" + "=" * 75)
print("TREINO vs TESTE — Gap R²: negativo = overfitting")
print("=" * 75)
display(df_comp.round(4))

# ── Plot barras duplas ─────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(21, 6))
nomes = list(df_treino.index)
x, w  = np.arange(len(nomes)), 0.35
bc    = [CORES[n] for n in nomes]

for ax, met in zip(axes, ['R²', 'RMSE', 'MAE']):
    tr_v, te_v = df_treino[met].values, df_teste[met].values
    b1 = ax.bar(x - w/2, tr_v, w, label='Treino', alpha=0.85, color=bc, edgecolor='white')
    b2 = ax.bar(x + w/2, te_v, w, label='Teste',  alpha=0.50, color=bc, edgecolor='black', linewidth=0.8)
    ax.set_xticks(x); ax.set_xticklabels(nomes, rotation=30, ha='right', fontsize=9)
    ax.set_title(f'{met}  Treino (sólido) vs Teste (transparente)', fontsize=10)
    ax.legend(fontsize=9); ax.grid(True, axis='y', alpha=0.3)
    if met == 'R²':
        ax.set_ylim(0, 1.05)
    for bar, v in zip(list(b1)+list(b2), list(tr_v)+list(te_v)):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+max(max(tr_v),max(te_v))*0.01,
                f'{v:.3f}', ha='center', fontsize=7.5)

plt.suptitle("Treino vs Teste — Diagnóstico de Overfitting", fontsize=13, y=1.02)
plt.tight_layout(); plt.show()

# ── Gap R² (semáforo) ─────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))
gaps = df_comp['Gap R² (Teste−Treino)'].values
gc   = ['#e74c3c' if g < -0.05 else '#f39c12' if g < -0.02 else '#2ecc71' for g in gaps]
bars = ax.barh(nomes, gaps, color=gc, alpha=0.85, edgecolor='white')
ax.axvline(0, color='black', lw=1)
ax.axvline(-0.05, color='orange', lw=1.5, linestyle='--', label='Limite aceitável (−0.05)')
ax.set_xlabel('Gap R² (Teste − Treino)  |  Próximo de 0 = sem overfitting', fontsize=10)
ax.set_title('Diagnóstico de Overfitting\n(verde: ok | laranja: moderado | vermelho: alto)', fontsize=11)
for bar, v in zip(bars, gaps):
    ax.text(v - 0.002 if v < 0 else v + 0.001, bar.get_y()+bar.get_height()/2,
            f'{v:.4f}', va='center', ha='right' if v < 0 else 'left', fontsize=10)
ax.legend(fontsize=9); ax.grid(True, axis='x', alpha=0.3)
plt.tight_layout(); plt.show()

---
## 6. Round 2 — Confiabilidade das Predições

### 6.1 Split Conformal Prediction

Garante cobertura nominal dos intervalos sem assumir distribuição dos erros. Ver `model_competition_notes.md §6`.

In [ ]:
# Retreinar em X_train (estado limpo para o conformal — remove outputs da célula anterior)
print("Retreinando todos os pipelines em X_train...")
for nome, pipe in pipelines_tunados.items():
    pipe.fit(X_train, y_train)
    r2_tr = r2_score(y_train, pipe.predict(X_train))
    r2_te = r2_score(y_test,  pipe.predict(X_test))
    print(f"  {nome:<12}: R²(treino)={r2_tr:.4f} | R²(teste)={r2_te:.4f}")
print("\nPipelines prontos para calibração conformal em X_cal.")

### 6.2 Avaliação com Intervalos de Confiança

In [ ]:
CONFIDENCE_LEVEL = 0.80
ALPHA            = 1 - CONFIDENCE_LEVEL
resultados_r2    = {}
intervalos       = {}
y_test_orig      = np.expm1(y_test.values)

for nome, pipe in pipelines_tunados.items():
    y_pred_log, lower_log, upper_log, q_hat = conformal_intervals(
        pipe, X_cal, y_cal, X_test, confidence=CONFIDENCE_LEVEL)
    y_pred_orig = np.expm1(y_pred_log)
    lower_orig  = np.maximum(np.expm1(lower_log), 0)
    upper_orig  = np.expm1(upper_log)

    is_val   = interval_score(y_test_orig, lower_orig, upper_orig, alpha=ALPHA)
    picp_val = picp(y_test_orig, lower_orig, upper_orig)
    mpiw_val = mpiw(lower_orig, upper_orig)
    mace_val = mace(pipe, X_cal, y_cal, X_test, y_test, levels=np.arange(0.1, 1.0, 0.1))

    intervalos[nome] = dict(y_pred_log=y_pred_log, lower_log=lower_log, upper_log=upper_log,
                             y_pred_orig=y_pred_orig, lower_orig=lower_orig,
                             upper_orig=upper_orig, q_hat=q_hat)

    resultados_r2[nome] = {
        'Modelo': nome,
        f'IS ({int(CONFIDENCE_LEVEL*100)}%)': is_val,
        f'PICP ({int(CONFIDENCE_LEVEL*100)}%)': picp_val,
        'PICP Error': abs(picp_val - CONFIDENCE_LEVEL),
        'MPIW ($100k)': mpiw_val,
        'MACE': mace_val,
        'q̂ (log)': q_hat,
    }
    print(f"{nome:<12}: IS={is_val:.4f} | PICP={picp_val:.4f} | MPIW={mpiw_val:.4f} | MACE={mace_val:.4f}")

df_r2 = pd.DataFrame(list(resultados_r2.values())).set_index('Modelo')
print(f"\n{'='*65}")
print(f"ROUND 2 — CONFIABILIDADE (Intervalo {int(CONFIDENCE_LEVEL*100)}%, Teste)")
print("=" * 65)
display(df_r2.round(4))

### 6.3 Curvas de Calibração

Cobertura empírica vs. nominal (10 %–90 %). Modelo calibrado segue a diagonal.

In [ ]:
levels = np.arange(0.10, 1.00, 0.10)
fig, ax = plt.subplots(figsize=(9, 7))
for nome, pipe in pipelines_tunados.items():
    cov_empiricas = []
    for c in levels:
        _, lo_log, hi_log, _ = conformal_intervals(pipe, X_cal, y_cal, X_test, confidence=c)
        cov_empiricas.append(picp(
            y_test_orig,
            np.maximum(np.expm1(lo_log), 0),
            np.expm1(hi_log),
        ))
    ax.plot(levels, cov_empiricas, marker='o', label=nome, color=CORES[nome], linewidth=2)
ax.plot([0, 1], [0, 1], 'k--', lw=1.5, label='Calibração perfeita', alpha=0.7)
ax.set_xlabel('Cobertura Nominal (1 − α)', fontsize=12)
ax.set_ylabel('Cobertura Empírica (PICP)', fontsize=12)
ax.set_title('Curva de Calibração — Conformal Prediction\n(mais próxima da diagonal = melhor)', fontsize=12)
ax.legend(fontsize=10); ax.set_xlim(0.05, 0.95); ax.set_ylim(0.05, 1.0)
ax.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout(); plt.show()

### 6.4 Visualização dos Intervalos de Predição

In [ ]:
np.random.seed(RANDOM_STATE)
idx_sample    = np.random.choice(len(y_test), size=100, replace=False)
y_test_sample = y_test_orig[idx_sample]
sorted_order  = np.argsort(y_test_sample)

fig, axes = plt.subplots(2, 3, figsize=(21, 12), sharey=True)
axes_flat  = axes.flatten()

for ax_idx, (nome, dados) in enumerate(intervalos.items()):
    ax   = axes_flat[ax_idx]
    lo   = dados['lower_orig'][idx_sample][sorted_order]
    hi   = dados['upper_orig'][idx_sample][sorted_order]
    pred = dados['y_pred_orig'][idx_sample][sorted_order]
    true = y_test_sample[sorted_order]
    x_r  = np.arange(len(lo))
    ax.fill_between(x_r, lo, hi, alpha=0.25, color=CORES[nome], label='Intervalo 80%')
    ax.plot(x_r, pred, color=CORES[nome], linewidth=1.5, label='Predição')
    ax.scatter(x_r, true, s=8, color='black', alpha=0.6, zorder=5, label='Real')
    ax.set_title(f'{nome}\nPICP={df_r2.loc[nome, "PICP (80%)"]:.3f} | IS={df_r2.loc[nome, "IS (80%)"]:.3f}',
                 fontsize=10)
    ax.set_xlabel('Amostra (ordenada por valor real)', fontsize=9)
    ax.set_ylabel('ValorMedioResidencias ($100k)', fontsize=9)
    ax.legend(fontsize=8)

axes_flat[-1].set_visible(False)
plt.suptitle("Intervalos de Predição 80% — Conformal Prediction (100 amostras do teste)", fontsize=13)
plt.tight_layout(); plt.show()

### 6.5 Resumo do Round 2

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(22, 6))
metricas_r2_def = [
    (f'IS ({int(CONFIDENCE_LEVEL*100)}%)',  'Interval Score (menor = melhor)',   True),
    ('PICP Error',  'Erro de Cobertura |PICP−0.8| (menor = melhor)', True),
    ('MPIW ($100k)', 'Largura do Intervalo (menor = melhor)', True),
    ('MACE',  'Calibration Error (menor = melhor)', True),
]
for ax, (col, label, lower_better) in zip(axes, metricas_r2_def):
    vals  = df_r2[col]
    best  = vals.idxmin() if lower_better else vals.idxmax()
    bc    = ['#2ecc71' if idx == best else CORES[idx] for idx in df_r2.index]
    bars  = ax.bar(df_r2.index, vals, color=bc, alpha=0.85, edgecolor='white', width=0.6)
    ax.set_title(label, fontsize=10)
    ax.set_xticklabels(df_r2.index, rotation=30, ha='right', fontsize=9)
    ax.grid(True, axis='y', alpha=0.3)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+max(vals)*0.01,
                f'{v:.4f}', ha='center', fontsize=8)
plt.suptitle("Round 2 — Métricas de Confiabilidade (verde = melhor)", fontsize=13, y=1.02)
plt.tight_layout(); plt.show()

---
## 7. Scoreboard Final

### 7.1 Ranking Ponderado dos Modelos

Round 1 (60 %): R² · RMSE · MAE · MAPE — Round 2 (40 %): IS · PICP Error · MACE. Ver `model_competition_notes.md §7`.

In [ ]:
def normalizar_metrica(series, maior_melhor=True):
    mi, ma = series.min(), series.max()
    if ma == mi:
        return pd.Series(1.0, index=series.index)
    norm = (series - mi) / (ma - mi)
    return norm if maior_melhor else 1 - norm

_IS_COL = f'IS ({int(CONFIDENCE_LEVEL*100)}%)'   # segue CONFIDENCE_LEVEL dinamicamente

PESOS = {
    'R² (CV)': (0.30, True), 'RMSE (CV)': (0.15, False),
    'MAE (CV)': (0.10, False), 'MAPE % (CV)': (0.05, False),
    _IS_COL: (0.20, False),
    'PICP Error': (0.10, False), 'MACE': (0.10, False),
}

df_score = df_r1.join(df_r2[[_IS_COL, 'PICP Error', 'MACE']])
df_norm  = pd.DataFrame(index=df_score.index)
for col, (peso, maior_melhor) in PESOS.items():
    if col in df_score.columns:
        df_norm[col] = normalizar_metrica(df_score[col], maior_melhor) * peso

df_score['Score Final'] = df_norm.sum(axis=1)
df_score['Posição']     = df_score['Score Final'].rank(ascending=False).astype(int)
df_score = df_score.sort_values('Score Final', ascending=False)

print("=" * 70)
print("SCOREBOARD FINAL — 5 Modelos (Round 1 + Round 2)")
print("=" * 70)
display(df_score[['R² (CV)', 'RMSE (CV)', 'MAPE % (CV)', _IS_COL, 'PICP Error', 'MACE', 'Score Final', 'Posição']].round(4))

vencedor = df_score.index[0]
print(f"\n{'='*70}")
print(f"  VENCEDOR: {vencedor}  |  Score Final: {df_score.loc[vencedor, 'Score Final']:.4f}")
print(f"{'='*70}")

### 7.2 Visualização do Ranking

In [ ]:
fig = plt.figure(figsize=(18, 9))
gs  = gridspec.GridSpec(2, 3, figure=fig)

ax_bar = fig.add_subplot(gs[:, 0])
medal  = {0: 'gold', 1: 'silver', 2: '#cd7f32'}
bc_bar = [medal.get(i, '#95a5a6') for i in range(len(df_score))]
bars   = ax_bar.barh(df_score.index[::-1], df_score['Score Final'][::-1],
                     color=bc_bar[::-1], alpha=0.9, edgecolor='white')
ax_bar.set_xlabel('Score Ponderado (0 = pior, 1 = melhor)', fontsize=10)
ax_bar.set_title('Score Final\n(Round 1 × 60% + Round 2 × 40%)', fontsize=11)
for bar, v in zip(bars, df_score['Score Final'][::-1]):
    ax_bar.text(v+0.005, bar.get_y()+bar.get_height()/2, f'{v:.4f}', va='center', fontsize=10)
ax_bar.set_xlim(0, 1); ax_bar.grid(True, axis='x', alpha=0.3)

df_norm_plot = pd.DataFrame(index=df_score.index)
for col, (_, mb) in PESOS.items():
    if col in df_score.columns:
        df_norm_plot[col] = normalizar_metrica(df_score[col], mb)

ax_heat = fig.add_subplot(gs[:, 1:])
im = ax_heat.imshow(df_norm_plot.values, cmap='RdYlGn', aspect='auto', vmin=0, vmax=1)
plt.colorbar(im, ax=ax_heat, label='Score Normalizado (0=pior, 1=melhor)', fraction=0.03, pad=0.04)
ax_heat.set_xticks(range(len(df_norm_plot.columns)))
ax_heat.set_xticklabels(df_norm_plot.columns, rotation=35, ha='right', fontsize=9)
ax_heat.set_yticks(range(len(df_norm_plot.index)))
ax_heat.set_yticklabels(df_norm_plot.index, fontsize=11)
ax_heat.set_title('Desempenho por Dimensão\n(verde = melhor, vermelho = pior)', fontsize=11)
for i in range(len(df_norm_plot.index)):
    for j in range(len(df_norm_plot.columns)):
        v = df_norm_plot.values[i, j]
        ax_heat.text(j, i, f'{v:.2f}', ha='center', va='center',
                     fontsize=9, color='black' if 0.3 < v < 0.7 else 'white')

plt.suptitle(f'Scoreboard Final — Vencedor: {vencedor}', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

---
## 8. Experimento Final — Modelo Vencedor

### 8.1 Protocolo

Retreinar em `X_trainval` (80 %) → avaliar em `X_test` → serializar.

In [ ]:
best_study = study_map[vencedor]

print(f"Modelo vencedor: {vencedor}")
print(f"Melhores hiperparâmetros (Optuna):")
for k, v in best_study.best_params.items():
    print(f"  {k}: {v}")
print(f"R² (CV) obtido: {best_study.best_value:.4f}")

# Treinar em X_train (60%) — X_cal permanece virgem para calibração conformal.
# Garantia Split Conformal exige exchangeability: modelo NÃO pode ter visto X_cal.
# Treinar em X_trainval violaria essa garantia e produziria q_hats subestimados.
pipeline_final = criar_pipeline(model_map[vencedor])
pipeline_final.fit(X_train, y_train)

print(f"\nPipeline final treinado em X_train ({X_train.shape[0]:,} amostras).")
print("X_cal reservado exclusivamente para calibração conformal (q_hats válidos).")

### 8.2 Avaliação Final no Conjunto de Teste

In [ ]:
y_pred_final_log = pipeline_final.predict(X_test)
m_ponto = metricas_ponto(y_test.values, y_pred_final_log, vencedor)
df_m_ponto = pd.DataFrame([m_ponto]).set_index('Modelo')

resultados_confiabilidade = []
for nivel in [0.80, 0.90]:
    _, low_log, up_log, _ = conformal_intervals(pipeline_final, X_cal, y_cal, X_test, confidence=nivel)
    low_orig = np.maximum(np.expm1(low_log), 0)
    up_orig  = np.expm1(up_log)
    y_t_orig = np.expm1(y_test.values)
    resultados_confiabilidade.append({
        'Nível': f'{int(nivel*100)}%',
        'PICP': picp(y_t_orig, low_orig, up_orig),
        'MPIW ($100k)': mpiw(low_orig, up_orig),
        'IS': interval_score(y_t_orig, low_orig, up_orig, alpha=1-nivel),
    })

df_confiab = pd.DataFrame(resultados_confiabilidade).set_index('Nível')
mace_final = mace(pipeline_final, X_cal, y_cal, X_test, y_test)

print("=" * 60)
print(f"AVALIAÇÃO FINAL — {vencedor} | Conjunto de Teste")
print("=" * 60)
print("\nMétricas de Ponto:")
display(df_m_ponto.round(4))
print(f"\nMétricas de Confiabilidade:")
display(df_confiab.round(4))
print(f"  MACE: {mace_final:.4f}  |  Erro típico: ${m_ponto['MAE ($100k)']*100_000:,.0f}")

# ─── Avaliação no dataset COMPLETO (in-sample) ───────────────────────────
print("\n" + "=" * 60)
print("AVALIAÇÃO IN-SAMPLE (todos os dados — apenas comparativo)")
print("AVISO: modelo treinado e avaliado nos MESMOS dados.")
print("Serve para ver o teto de desempenho, não generalização.")
print("=" * 60)

import copy
y_all = np.log1p(df_raw[TARGET])
X_all = df_raw[FEATURES]

resultados_total = []
for nome, pipe_orig in pipelines_tunados.items():
    pipe_total = copy.deepcopy(pipe_orig)
    pipe_total.fit(X_all, y_all)
    m = metricas_ponto(y_all.values, pipe_total.predict(X_all), nome)
    resultados_total.append(m)

df_total = pd.DataFrame(resultados_total).set_index('Modelo')
print("\nMétricas In-sample (dataset completo):")
display(df_total.round(4))

# ── Plot: CV Treino vs In-sample Total ────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
nomes_t = list(df_total.index)
x, w    = np.arange(len(nomes_t)), 0.35
bc      = [CORES[n] for n in nomes_t]

for ax, (col_cv, col_is, label) in zip(axes, [
    ('R² (CV)', 'R²', 'R²'),
    ('RMSE (CV)', 'RMSE ($100k)', 'RMSE'),
]):
    cv_v = df_r1[col_cv].reindex(nomes_t).values
    is_v = df_total[col_is].values
    b1 = ax.bar(x - w/2, cv_v, w, label='CV Treino', alpha=0.55, color=bc, edgecolor='black', lw=0.8)
    b2 = ax.bar(x + w/2, is_v, w, label='In-sample Total', alpha=0.88, color=bc, edgecolor='white')
    ax.set_xticks(x); ax.set_xticklabels(nomes_t, rotation=30, ha='right')
    ax.set_title(f'{label}: CV Treino vs In-sample Total', fontsize=11)
    ax.legend(); ax.grid(True, axis='y', alpha=0.3)
    if label == 'R²': ax.set_ylim(0, 1.05)
    for bar, v in zip(list(b1)+list(b2), list(cv_v)+list(is_v)):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+max(max(cv_v),max(is_v))*0.01,
                f'{v:.3f}', ha='center', fontsize=8)

plt.suptitle("CV Treino vs In-sample Total\n(gap grande = overfitting | in-sample sempre melhor)", fontsize=12)
plt.tight_layout(); plt.show()

### 8.3 Visualizações do Modelo Vencedor

In [ ]:
# ─── Plots finais ──────────────────────────────────────────────────────────
_, low80_log, up80_log, _ = conformal_intervals(pipeline_final, X_cal, y_cal, X_test, confidence=0.80)
_, low90_log, up90_log, _ = conformal_intervals(pipeline_final, X_cal, y_cal, X_test, confidence=0.90)

y_pred_usd  = np.expm1(y_pred_final_log) * 100  # $1k USD
y_test_usd  = np.expm1(y_test.values)    * 100
low80_usd   = np.maximum(np.expm1(low80_log), 0) * 100
up80_usd    = np.expm1(up80_log) * 100
low90_usd   = np.maximum(np.expm1(low90_log), 0) * 100
up90_usd    = np.expm1(up90_log) * 100

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Real vs Previsto
axes[0].scatter(y_test_usd, y_pred_usd, alpha=0.2, s=4, c='steelblue')
lim = max(y_test_usd.max(), y_pred_usd.max())
axes[0].plot([0, lim], [0, lim], 'r--', lw=1.5, label='Perfeito')
axes[0].set_xlabel('Valor Real (USD mil)')
axes[0].set_ylabel('Valor Previsto (USD mil)')
axes[0].set_title(f'Real vs Previsto — {vencedor}')
axes[0].legend()

# Intervalos ordenados (150 amostras)
np.random.seed(RANDOM_STATE)
idx = np.random.choice(len(y_test_usd), 150, replace=False)
order = np.argsort(y_test_usd[idx])
x_r = np.arange(150)
axes[1].fill_between(x_r, low90_usd[idx][order], up90_usd[idx][order], alpha=0.15, color='orange', label='90% CI')
axes[1].fill_between(x_r, low80_usd[idx][order], up80_usd[idx][order], alpha=0.30, color='steelblue', label='80% CI')
axes[1].plot(x_r, y_pred_usd[idx][order], color='steelblue', lw=1.5, label='Predição')
axes[1].scatter(x_r, y_test_usd[idx][order], s=10, color='black', alpha=0.7, zorder=5, label='Real')
axes[1].set_xlabel('Amostras (ordenadas por valor real)')
axes[1].set_ylabel('ValorMedioResidencias (USD mil)')
axes[1].set_title(f'Intervalos de Predição Finais — {vencedor}')
axes[1].legend(fontsize=9)

plt.suptitle(f'Experimento Final — Modelo: {vencedor}', fontsize=14)
plt.tight_layout()
plt.show()

### 8.4 Serialização e Artefatos

Pipeline (`joblib`) + metadados JSON com `q_hats` conformal. Ver `model_competition_notes.md §8.4`.

In [ ]:
MODEL_PATH    = ARTIFACT_DIR / f'competition_winner_{vencedor.lower()}.joblib'
METADATA_PATH = ARTIFACT_DIR / 'competition_metadata.json'

joblib.dump(pipeline_final, MODEL_PATH, compress=3)

q_hats = {}
for nivel in [0.80, 0.90, 0.95]:
    _, _, _, q = conformal_intervals(pipeline_final, X_cal, y_cal, X_test, confidence=nivel)
    q_hats[str(nivel)] = round(q, 6)

metadata = {
    'competition_winner' : vencedor,
    'competing_models'   : list(pipelines_tunados.keys()),
    'tuner'              : 'Optuna TPESampler (multivariate=True) + MedianPruner (boosting)',
    'n_trials'           : {'Ridge': TARGET_TRIALS['CalHousing_Ridge'], 'XGBoost': TARGET_TRIALS['CalHousing_XGBoost'],
                            'LightGBM': TARGET_TRIALS['CalHousing_LightGBM'], 'CatBoost': TARGET_TRIALS['CalHousing_CatBoost'],
                            'TabNet': TARGET_TRIALS['CalHousing_TabNet']},
    'gpu_used'           : CUDA_AVAILABLE,
    'device'             : DEVICE,
    'training_date'      : datetime.now().isoformat(),
    'sklearn_version'    : sklearn.__version__,
    'optuna_version'     : optuna.__version__,
    'python_version'     : sys.version.split()[0],
    'n_train_samples'    : int(X_train.shape[0]),
    'n_cal_samples'      : int(X_cal.shape[0]),
    'n_test_samples'     : int(X_test.shape[0]),
    'input_features'     : FEATURES,
    'engineered_features': CaliforniaHousingTransformer.OUTPUT_COLS,
    'target_column'      : TARGET,
    'target_transform'   : 'np.log1p — reverter com np.expm1',
    'target_unit'        : '$100k USD',
    'conformal_q_hats'   : q_hats,
    'best_hyperparams'   : study_map[vencedor].best_params,
    'best_cv_r2'         : round(study_map[vencedor].best_value, 4),
    'all_scores'         : {m: round(float(df_score.loc[m, 'Score Final']), 4)
                            for m in df_score.index},
    'metrics_test': {
        'R2'        : round(m_ponto['R²'], 4),
        'RMSE_100k' : round(m_ponto['RMSE ($100k)'], 4),
        'MAE_100k'  : round(m_ponto['MAE ($100k)'], 4),
        'MAPE_pct'  : round(m_ponto['MAPE (%)'], 2),
        'IS_80pct'  : round(float(df_confiab.loc['80%', 'IS']), 4),
        'PICP_80pct': round(float(df_confiab.loc['80%', 'PICP']), 4),
        'MACE'      : round(mace_final, 4),
    },
}

with open(METADATA_PATH, 'w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

size_mb = MODEL_PATH.stat().st_size / 1_048_576
print(f"Pipeline salvo : {MODEL_PATH}  ({size_mb:.2f} MB)")
print(f"Metadados      : {METADATA_PATH}")
for nivel, q in q_hats.items():
    print(f"  {float(nivel)*100:.0f}% → q̂ = {q:.4f}")

---
## 9. Inferência

### 9.1 Função de Inferência

In [ ]:
def carregar_e_prever(
    pipeline_path,
    metadata_path,
    input_data,
    confidence=0.80,
):
    """
    Carrega o pipeline serializado e realiza inferência com intervalo de confiança.

    O intervalo é construído usando os q̂ pré-calculados do conformal prediction
    (armazenados no metadata.json). Para um novo conjunto de calibração, use
    conformal_intervals() diretamente.

    Parâmetros
    ----------
    pipeline_path : str | Path
        Caminho para o arquivo .joblib do pipeline.
    metadata_path : str | Path
        Caminho para o metadata.json (contém q̂ de calibração).
    input_data : dict | pd.DataFrame
        Um ou mais registros com features originais (sem target).
    confidence : float
        Nível de confiança para o intervalo (0.80, 0.90 ou 0.95).

    Retorna
    -------
    list[dict] com 'predicted_100k', 'predicted_usd', 'lower_usd', 'upper_usd'
    """
    pipe = joblib.load(pipeline_path)
    with open(metadata_path, 'r') as f:
        meta = json.load(f)

    q_hat = meta['conformal_q_hats'].get(str(confidence))
    if q_hat is None:
        available = list(meta['conformal_q_hats'].keys())
        raise ValueError(f"confidence={confidence} não disponível. Use: {available}")

    df_input = pd.DataFrame([input_data]) if isinstance(input_data, dict) else pd.DataFrame(input_data)

    pred_log    = pipe.predict(df_input)
    lower_log   = pred_log - q_hat
    upper_log   = pred_log + q_hat

    resultados = []
    for p, lo, hi in zip(pred_log, lower_log, upper_log):
        resultados.append({
            'predicted_100k' : round(float(np.expm1(p)), 4),
            'predicted_usd'  : round(float(np.expm1(p) * 100_000), 0),
            'lower_usd'      : round(float(max(np.expm1(lo) * 100_000, 0)), 0),
            'upper_usd'      : round(float(np.expm1(hi) * 100_000), 0),
            'confidence'     : confidence,
        })
    return resultados[0] if len(resultados) == 1 else resultados

print("Função 'carregar_e_prever' definida com suporte a intervalos de confiança.")

### 9.2 Exemplos de Predição

In [ ]:
# ─── Exemplos de inferência com diferentes perfis ─────────────────────────
exemplos = [
    ('Los Angeles (renda média)',
     dict(RendaMediana=5.0, IdadeMediaResidencias=20.0, MediaComodos=5.5,
          MediaQuartos=1.1, Populacao=1200.0, MediaOcupacao=2.8,
          Latitude=34.05, Longitude=-118.24)),
    ('Bay Area (alta renda)',
     dict(RendaMediana=10.0, IdadeMediaResidencias=35.0, MediaComodos=6.0,
          MediaQuartos=1.05, Populacao=900.0, MediaOcupacao=2.5,
          Latitude=37.77, Longitude=-122.42)),
    ('Interior Califórnia (baixa renda)',
     dict(RendaMediana=2.5, IdadeMediaResidencias=15.0, MediaComodos=4.5,
          MediaQuartos=1.2, Populacao=2000.0, MediaOcupacao=3.5,
          Latitude=36.77, Longitude=-119.79)),
]

print(f"Inferência com {vencedor} — Intervalos Conformal (80% e 90%):\n")
print(f"{'Região':<35} {'Predição':>12} {'IC 80%':>30} {'IC 90%':>30}")
print("-" * 110)

for nome_ex, amostra in exemplos:
    r80 = carregar_e_prever(MODEL_PATH, METADATA_PATH, amostra, confidence=0.80)
    r90 = carregar_e_prever(MODEL_PATH, METADATA_PATH, amostra, confidence=0.90)
    ic80 = f"[${r80['lower_usd']:>9,.0f} — ${r80['upper_usd']:>9,.0f}]"
    ic90 = f"[${r90['lower_usd']:>9,.0f} — ${r90['upper_usd']:>9,.0f}]"
    print(f"{nome_ex:<35} ${r80['predicted_usd']:>10,.0f}  {ic80:>30}  {ic90:>30}")

print("\nConsistência pipeline salvo vs memória:")
for _, amostra in exemplos[:1]:
    pred_mem   = float(np.expm1(pipeline_final.predict(pd.DataFrame([amostra])))[0])
    pred_disco = carregar_e_prever(MODEL_PATH, METADATA_PATH, amostra)['predicted_100k']
    ok = abs(pred_mem - pred_disco) < 1e-6
    print(f"  Diferença: {abs(pred_mem - pred_disco):.2e} → {'OK' if ok else 'ERRO'}")

---
## 10. Conclusão

### 10.1 Resultados da Competição

Resultados esperados, análise de GPU e próximos passos: ver `model_competition_notes.md §10`.